## 02 – Tạo Lag Features và tách X, y
Mục tiêu: Chuyển chuỗi thời gian đơn biến AEP_MW thành dữ liệu học có giám sát bằng các biến trễ.

Notebook thực hiện:
1. Đọc data/raw/AEP_hourly.csv.
2. Chuyển Datetime sang kiểu datetime, kiểm tra missing values và timestamp trùng.
3. Xử lý timestamp trùng bằng cách lấy trung bình AEP_MW tại cùng thời điểm; sắp xếp tăng dần và đặt Datetime làm index.
4. Tạo lag_1 đến lag_p.
5. Loại bỏ các dòng thiếu do thao tác shift.
6. Tạo X từ các cột lag và y từ AEP_MW hiện tại.
7. Kiểm tra và lưu dữ liệu.

# 1. Import thư viện và đọc dữ liệu

In [19]:
from pathlib import Path
import pandas as pd
import numpy as np

# Hỗ trợ chạy notebook từ thư mục gốc repo hoặc từ thư mục notebooks/
candidate_paths = [
    Path("data/raw/AEP_hourly.csv"),
    Path("../data/raw/AEP_hourly.csv")
]
data_path = next((p for p in candidate_paths if p.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "Không tìm thấy data/raw/AEP_hourly.csv. "
        "Hãy đặt AEP_hourly.csv trong thư mục data/raw/ của repository."
    )

df = pd.read_csv(data_path)
print("Đường dẫn dữ liệu:", data_path)
print("Kích thước ban đầu:", df.shape)
display(df.head())
display(df.info())

Đường dẫn dữ liệu: ..\data\raw\AEP_hourly.csv
Kích thước ban đầu: (121273, 2)


,Datetime,AEP_MW
0,2004-12-31 01:00:00,13478.0
1,2004-12-31 02:00:00,12865.0
2,2004-12-31 03:00:00,12577.0
3,2004-12-31 04:00:00,12517.0
4,2004-12-31 05:00:00,12670.0


<class 'pandas.DataFrame'>
RangeIndex: 121273 entries, 0 to 121272
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   Datetime  121273 non-null  str    
 1   AEP_MW    121273 non-null  float64
dtypes: float64(1), str(1)
memory usage: 4.0 MB


None

# 2. Chuyển Datetime và kiểm tra chất lượng dữ liệu

In [20]:
df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")

print("Kiểu dữ liệu sau chuyển đổi:")
print(df.dtypes)

print("\\nSố giá trị thiếu theo cột:")
print(df.isna().sum())

print("\\nSố Datetime không hợp lệ:", df["Datetime"].isna().sum())
print("Số dòng thuộc các timestamp bị trùng:",
      df.duplicated(subset=["Datetime"], keep=False).sum())
print("Số timestamp trùng (tính số lần xuất hiện dư):",
      df["Datetime"].duplicated().sum())

duplicate_detail = (
    df[df.duplicated(subset=["Datetime"], keep=False)]
    .sort_values("Datetime")
)
display(duplicate_detail)

Kiểu dữ liệu sau chuyển đổi:
Datetime    datetime64[us]
AEP_MW             float64
dtype: object
\nSố giá trị thiếu theo cột:
Datetime    0
AEP_MW      0
dtype: int64
\nSố Datetime không hợp lệ: 0
Số dòng thuộc các timestamp bị trùng: 8
Số timestamp trùng (tính số lần xuất hiện dư): 4


,Datetime,AEP_MW
82492,2014-11-02 02:00:00,12994.0
82493,2014-11-02 02:00:00,13190.0
91275,2015-11-01 02:00:00,10785.0
91276,2015-11-01 02:00:00,10542.0
99915,2016-11-06 02:00:00,10964.0
99916,2016-11-06 02:00:00,11008.0
108723,2017-11-05 02:00:00,10596.0
108724,2017-11-05 02:00:00,10446.0


# 3. Xử lý timestamp trùng, sắp xếp thời gian và đặt index

In [21]:
# Bỏ các dòng có Datetime không hợp lệ (nếu có)
df = df.dropna(subset=["Datetime"])

# Gộp các timestamp trùng bằng trung bình
data = (
    df.groupby("Datetime", as_index=False)["AEP_MW"].mean()
      .sort_values("Datetime")
      .set_index("Datetime")
)

print("Kích thước sau xử lý timestamp trùng:", data.shape)
print("Index tăng dần:", data.index.is_monotonic_increasing)
print("Index còn trùng:", data.index.has_duplicates)
print("Missing values sau xử lý:")
print(data.isna().sum())

# Kiểm tra khoảng cách giữa các timestamp
time_gaps = data.index.to_series().diff().value_counts().sort_index()
print("\\nCác khoảng cách thời gian thường gặp:")
display(time_gaps.head(10))
print("Số khoảng cách lớn hơn 1 giờ:",
      int((data.index.to_series().diff() > pd.Timedelta(hours=1)).sum()))

display(data.head())

Kích thước sau xử lý timestamp trùng: (121269, 1)
Index tăng dần: True
Index còn trùng: False
Missing values sau xử lý:
AEP_MW    0
dtype: int64
\nCác khoảng cách thời gian thường gặp:


Datetime
0 days 01:00:00    121241
0 days 02:00:00        27
Name: count, dtype: int64

Số khoảng cách lớn hơn 1 giờ: 27


,AEP_MW
Datetime,
2004-10-01 01:00:00,12379.0
2004-10-01 02:00:00,11935.0
2004-10-01 03:00:00,11692.0
2004-10-01 04:00:00,11597.0
2004-10-01 05:00:00,11681.0


# 4. Chọn p và viết hàm create_lag_features
Hàm dùng shift(i) để tạo lag_i = AEP_MW của i quan sát trước đó.

In [22]:
def create_lag_features(data, target_col, p):
    """Tạo các biến trễ lag_1 ... lag_p cho target_col.

    Parameters
    ----------
    data : pandas.DataFrame
        Dữ liệu chuỗi thời gian, đã sắp xếp tăng dần theo thời gian.
    target_col : str
        Tên cột mục tiêu.
    p : int
        Số độ trễ cần tạo.

    Returns
    -------
    pandas.DataFrame
        DataFrame gồm target_col hiện tại và p cột lag.
    """
    if target_col not in data.columns:
        raise KeyError(f"Không tìm thấy cột {target_col!r}")
    if not isinstance(p, int) or p < 1:
        raise ValueError("p phải là số nguyên dương")
    if not data.index.is_monotonic_increasing:
        raise ValueError("DataFrame phải được sắp xếp theo thời gian tăng dần")

    result = data[[target_col]].copy()
    for i in range(1, p + 1):
        result[f"lag_{i}"] = data[target_col].shift(i)

    return result

p = 24
lagged_data = create_lag_features(data, target_col="AEP_MW", p=p)
display(lagged_data.head(27))

,AEP_MW,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,lag_9,...,lag_15,lag_16,lag_17,lag_18,lag_19,lag_20,lag_21,lag_22,lag_23,lag_24
Datetime,,,,,,,,,,,,,,,,,,,,,
2004-10-01 01:00:00,12379.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2004-10-01 02:00:00,11935.0,12379.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2004-10-01 03:00:00,11692.0,11935.0,12379.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2004-10-01 04:00:00,11597.0,11692.0,11935.0,12379.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2004-10-01 05:00:00,11681.0,11597.0,11692.0,11935.0,12379.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2004-10-01 06:00:00,12280.0,11681.0,11597.0,11692.0,11935.0,12379.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2004-10-01 07:00:00,13692.0,12280.0,11681.0,11597.0,11692.0,11935.0,12379.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2004-10-01 08:00:00,14618.0,13692.0,12280.0,11681.0,11597.0,11692.0,11935.0,12379.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2004-10-01 09:00:00,14903.0,14618.0,13692.0,12280.0,11681.0,11597.0,11692.0,11935.0,12379.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 5. Loại bỏ dòng thiếu sau khi tạo lag

In [23]:
print("Missing values trước dropna:")
print(lagged_data.isna().sum())

lagged_data = lagged_data.dropna().copy()

print("Kích thước sau dropna:", lagged_data.shape)
print("Missing values sau dropna:")
print(lagged_data.isna().sum())

Missing values trước dropna:
AEP_MW     0
lag_1      1
lag_2      2
lag_3      3
lag_4      4
lag_5      5
lag_6      6
lag_7      7
lag_8      8
lag_9      9
lag_10    10
lag_11    11
lag_12    12
lag_13    13
lag_14    14
lag_15    15
lag_16    16
lag_17    17
lag_18    18
lag_19    19
lag_20    20
lag_21    21
lag_22    22
lag_23    23
lag_24    24
dtype: int64
Kích thước sau dropna: (121245, 25)
Missing values sau dropna:
AEP_MW    0
lag_1     0
lag_2     0
lag_3     0
lag_4     0
lag_5     0
lag_6     0
lag_7     0
lag_8     0
lag_9     0
lag_10    0
lag_11    0
lag_12    0
lag_13    0
lag_14    0
lag_15    0
lag_16    0
lag_17    0
lag_18    0
lag_19    0
lag_20    0
lag_21    0
lag_22    0
lag_23    0
lag_24    0
dtype: int64


# 6. Tạo ma trận X và vector y

In [24]:
lag_columns = [f"lag_{i}" for i in range(1, p + 1)]

X = lagged_data[lag_columns].copy()
y = lagged_data["AEP_MW"].copy()

print("Kích thước X:", X.shape)
print("Kích thước y:", y.shape)
print("\\nMissing values trong X:", X.isna().sum().sum())
print("Missing values trong y:", y.isna().sum())
print("\\nCác cột của X:")
print(X.columns.tolist())

display(X.head())
display(y.head())

Kích thước X: (121245, 24)
Kích thước y: (121245,)
\nMissing values trong X: 0
Missing values trong y: 0
\nCác cột của X:
['lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'lag_8', 'lag_9', 'lag_10', 'lag_11', 'lag_12', 'lag_13', 'lag_14', 'lag_15', 'lag_16', 'lag_17', 'lag_18', 'lag_19', 'lag_20', 'lag_21', 'lag_22', 'lag_23', 'lag_24']


,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,lag_9,lag_10,...,lag_15,lag_16,lag_17,lag_18,lag_19,lag_20,lag_21,lag_22,lag_23,lag_24
Datetime,,,,,,,,,,,,,,,,,,,,,
2004-10-02 01:00:00,13147.0,14067.0,14837.0,15349.0,15211.0,15034.0,15353.0,15644.0,15739.0,15739.0,...,15118.0,14903.0,14618.0,13692.0,12280.0,11681.0,11597.0,11692.0,11935.0,12379.0
2004-10-02 02:00:00,12260.0,13147.0,14067.0,14837.0,15349.0,15211.0,15034.0,15353.0,15644.0,15739.0,...,15242.0,15118.0,14903.0,14618.0,13692.0,12280.0,11681.0,11597.0,11692.0,11935.0
2004-10-02 03:00:00,11672.0,12260.0,13147.0,14067.0,14837.0,15349.0,15211.0,15034.0,15353.0,15644.0,...,15375.0,15242.0,15118.0,14903.0,14618.0,13692.0,12280.0,11681.0,11597.0,11692.0
2004-10-02 04:00:00,11352.0,11672.0,12260.0,13147.0,14067.0,14837.0,15349.0,15211.0,15034.0,15353.0,...,15404.0,15375.0,15242.0,15118.0,14903.0,14618.0,13692.0,12280.0,11681.0,11597.0
2004-10-02 05:00:00,11177.0,11352.0,11672.0,12260.0,13147.0,14067.0,14837.0,15349.0,15211.0,15034.0,...,15655.0,15404.0,15375.0,15242.0,15118.0,14903.0,14618.0,13692.0,12280.0,11681.0


Datetime
2004-10-02 01:00:00    12260.0
2004-10-02 02:00:00    11672.0
2004-10-02 03:00:00    11352.0
2004-10-02 04:00:00    11177.0
2004-10-02 05:00:00    11142.0
Name: AEP_MW, dtype: float64

# 7. Lưu dữ liệu sau khi tạo lag

In [16]:
# Khi chạy từ notebooks/, lưu vào ../data/processed/;
# khi chạy từ thư mục gốc repo, lưu vào data/processed/.
if Path("data").exists():
    output_dir = Path("data/processed")
else:
    output_dir = Path("../data/processed")

output_dir.mkdir(parents=True, exist_ok=True)

lagged_data.to_csv(output_dir / f"AEP_lag_features_p{p}.csv",
                   index_label="Datetime")
X.to_csv(output_dir / f"X_lag_p{p}.csv", index_label="Datetime")
y.to_frame().to_csv(output_dir / "y_AEP_MW.csv", index_label="Datetime")

print("Đã lưu dữ liệu tại:", output_dir.resolve())
print(" -", f"AEP_lag_features_p{p}.csv")
print(" -", f"X_lag_p{p}.csv")
print(" - y_AEP_MW.csv")

Đã lưu dữ liệu tại: C:\03_time\xgboost-time-series-lab2\DATA\processed
 - AEP_lag_features_p24.csv
 - X_lag_p24.csv
 - y_AEP_MW.csv


# 8. Ghi chú kết quả
- p = 24: sử dụng 24 giá trị tiêu thụ điện trước đó làm đặc trưng.
- X: gồm 24 cột lag_1 đến lag_24.
- y: là giá trị AEP_MW tại thời điểm hiện tại.
- 24 dòng đầu tiên bị loại vì không đủ lịch sử để tạo đủ 24 lag.
- Các timestamp trùng được gộp bằng trung bình; dữ liệu vẫn còn một số khoảng cách 2 giờ sau khi xử lý. Vì vậy, lag được hiểu là 24 quan sát trước đó, không đảm bảo luôn là đúng 24 giờ đồng hồ trước đó tại các đoạn có giờ khuyết.

## 9.Chia ma trận đặt trưng X và đặt trưng Y

- `X`: gồm các cột `lag_1` đến `lag_24`, dùng làm dữ liệu đầu vào cho mô hình.
- `y`: gồm cột `AEP_MW` tại thời điểm hiện tại, là giá trị mô hình cần dự đoán.
- X và y giữ cùng chỉ mục thời gian để đảm bảo mỗi hàng đầu vào tương ứng đúng với giá trị mục tiêu.

In [17]:
# X gồm các biến trễ lag_1, lag_2, ..., lag_24
lag_columns = [f"lag_{i}" for i in range(1, p + 1)]

X = lagged_data[lag_columns].copy()

# y là giá trị AEP_MW tại thời điểm hiện tại cần dự đoán
y = lagged_data["AEP_MW"].copy()

# Kiểm tra kích thước và tính đầy đủ của dữ liệu
print("Số dòng và số cột của X:", X.shape)
print("Số phần tử của y:", y.shape)
print("Số đặc trưng đầu vào:", X.shape[1])
print("Số giá trị thiếu trong X:", X.isna().sum().sum())
print("Số giá trị thiếu trong y:", y.isna().sum())
print("X và y có cùng chỉ mục thời gian:", X.index.equals(y.index))

print("\nDanh sách các cột đầu vào X:")
print(X.columns.tolist())

print("\nMột số dòng đầu tiên của X:")
display(X.head())

print("\nMột số giá trị đầu tiên của y:")
display(y.head())

# Đảm bảo dữ liệu sẵn sàng cho bước huấn luyện mô hình
assert X.index.equals(y.index), "Chỉ mục thời gian của X và y không khớp!"
assert X.isna().sum().sum() == 0, "X vẫn còn giá trị thiếu!"
assert y.isna().sum() == 0, "y vẫn còn giá trị thiếu!"

Số dòng và số cột của X: (121245, 24)
Số phần tử của y: (121245,)
Số đặc trưng đầu vào: 24
Số giá trị thiếu trong X: 0
Số giá trị thiếu trong y: 0
X và y có cùng chỉ mục thời gian: True

Danh sách các cột đầu vào X:
['lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'lag_8', 'lag_9', 'lag_10', 'lag_11', 'lag_12', 'lag_13', 'lag_14', 'lag_15', 'lag_16', 'lag_17', 'lag_18', 'lag_19', 'lag_20', 'lag_21', 'lag_22', 'lag_23', 'lag_24']

Một số dòng đầu tiên của X:


,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,lag_9,lag_10,...,lag_15,lag_16,lag_17,lag_18,lag_19,lag_20,lag_21,lag_22,lag_23,lag_24
Datetime,,,,,,,,,,,,,,,,,,,,,
2004-10-02 01:00:00,13147.0,14067.0,14837.0,15349.0,15211.0,15034.0,15353.0,15644.0,15739.0,15739.0,...,15118.0,14903.0,14618.0,13692.0,12280.0,11681.0,11597.0,11692.0,11935.0,12379.0
2004-10-02 02:00:00,12260.0,13147.0,14067.0,14837.0,15349.0,15211.0,15034.0,15353.0,15644.0,15739.0,...,15242.0,15118.0,14903.0,14618.0,13692.0,12280.0,11681.0,11597.0,11692.0,11935.0
2004-10-02 03:00:00,11672.0,12260.0,13147.0,14067.0,14837.0,15349.0,15211.0,15034.0,15353.0,15644.0,...,15375.0,15242.0,15118.0,14903.0,14618.0,13692.0,12280.0,11681.0,11597.0,11692.0
2004-10-02 04:00:00,11352.0,11672.0,12260.0,13147.0,14067.0,14837.0,15349.0,15211.0,15034.0,15353.0,...,15404.0,15375.0,15242.0,15118.0,14903.0,14618.0,13692.0,12280.0,11681.0,11597.0
2004-10-02 05:00:00,11177.0,11352.0,11672.0,12260.0,13147.0,14067.0,14837.0,15349.0,15211.0,15034.0,...,15655.0,15404.0,15375.0,15242.0,15118.0,14903.0,14618.0,13692.0,12280.0,11681.0



Một số giá trị đầu tiên của y:


Datetime
2004-10-02 01:00:00    12260.0
2004-10-02 02:00:00    11672.0
2004-10-02 03:00:00    11352.0
2004-10-02 04:00:00    11177.0
2004-10-02 05:00:00    11142.0
Name: AEP_MW, dtype: float64

## 10. Lưu dữ liệu sau khi tạo lag và chia X – y

In [18]:
# Khi chạy từ notebooks/, lưu vào ../data/processed/;
# khi chạy từ thư mục gốc repo, lưu vào data/processed/.
if Path("data").exists():
    output_dir = Path("data/processed")
else:
    output_dir = Path("../data/processed")

output_dir.mkdir(parents=True, exist_ok=True)

lagged_data.to_csv(output_dir / f"AEP_lag_features_p{p}.csv",
                   index_label="Datetime")
X.to_csv(output_dir / f"X_lag_p{p}.csv", index_label="Datetime")
y.to_frame().to_csv(output_dir / "y_AEP_MW.csv", index_label="Datetime")

print("Đã lưu dữ liệu tại:", output_dir.resolve())
print(" -", f"AEP_lag_features_p{p}.csv")
print(" -", f"X_lag_p{p}.csv")
print(" - y_AEP_MW.csv")

Đã lưu dữ liệu tại: C:\03_time\xgboost-time-series-lab2\DATA\processed
 - AEP_lag_features_p24.csv
 - X_lag_p24.csv
 - y_AEP_MW.csv


## 5. Ghi chú kết quả
- p = 24: sử dụng 24 giá trị tiêu thụ điện trước đó làm đặc trưng.
- X: gồm 24 cột lag_1 đến lag_24.
- y: là giá trị AEP_MW tại thời điểm hiện tại.
- 24 dòng đầu tiên bị loại vì không đủ lịch sử để tạo đủ 24 lag.
- Các timestamp trùng được gộp bằng trung bình; dữ liệu vẫn còn một số khoảng cách 2 giờ sau khi xử lý. Vì vậy, lag được hiểu là 24 quan sát trước đó, không đảm bảo luôn là đúng 24 giờ đồng hồ trước đó tại các đoạn có giờ khuyết.
- X và y có cùng chỉ mục thời gian, không còn giá trị thiếu và có thể được sử dụng ở Bước 4 để chia Train/Test theo thứ tự thời gian.